代码生成的输入是AST，目标是IR，所以我们接下来要做的工作其实很Naive，即遍历我们的AST，然后不断的吐出我们的IR指令即可，与此同时，每一个IR都会有自己的IR范式，这些功能都可以通过IR的IRBuilder来做到，于是我们的工作就变为了遍历AST，然后使用IRBuilder构建等价的（优化的）IR出来。Python则为这个提供了很方便的一个基类ast.NodeVisitor，我们只需要继承这个基类，然后override visit_XXX即可以完成我们的目的。

In [ ]:
import inspect
import ast

def jit(target="cpu"):
    assert target in ["cpu", "gpu"]
    def inner(fn):
        return JIT(fn, target=target)
    return inner

class JIT:
    def __init__(self, fn, target="cpu"):
        self.fn = fn
        self.target = target
    
    def __call__(self, *args, **kwargs):
        fn_src = inspect.getsource(self.fn)
        fn_ast = ast.parse(fn_src)
        print(ast.dump(fn_ast))
        code_generator = CodeGenerator(fn_ast, self.target)
        code_generator.code_gen()

class CodeGenerator(ast.NodeVisitor):
    def __init__(self, fn_ast, target):
        self.fn_ast = fn_ast
        self.target = target
    
    def code_gen(self):
        self.visit(self.fn_ast)

    def visit(self, node):
        print("Visit " + node.__class__.__name__)
        return super().visit(node)

@jit(target="cpu")
def add():
    print("add")

add()

Module(body=[FunctionDef(name='add', args=arguments(posonlyargs=[], args=[], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[Expr(value=Call(func=Name(id='print', ctx=Load()), args=[Constant(value='add', kind=None)], keywords=[]))], decorator_list=[Call(func=Name(id='jit', ctx=Load()), args=[], keywords=[keyword(arg='target', value=Constant(value='cpu', kind=None))])], returns=None, type_comment=None)], type_ignores=[])
Visit Module
1
Visit FunctionDef
2
Visit arguments
3
Visit Expr
4
Visit Call
5
Visit Name
6
Visit Load
7
Visit Constant
8
Visit Call
9
Visit Name
10
Visit Load
11
Visit keyword
12
Visit Constant
13
